# Entropy Diagnostics in GNNs
notebook for reproducing all results and figures.


In [ ]:
# Cell 2: Install & Setup
# ============================================================
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-2.5.0+cu124.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-2.5.0+cu124.html
!pip install -q torch-geometric
!pip install -q mycolorpy networkx

import importlib.util, sys, os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde, spearmanr

sns.set_theme(style="whitegrid", font_scale=1.05)
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

!git clone https://github.com/shanzi-bao/project.git /content/project

gcn_mod = load_module('gcn', '/content/project/gcn.py')
gat_mod = load_module('gat', '/content/project/gat.py')
linear_probe = load_module('linear_probe', '/content/project/linear_probe.py')
my_datasets = load_module('my_datasets', '/content/project/my_datasets.py')
synthetic_dataset = load_module('synthetic_dataset', '/content/project/synthetic_dataset.py')
pm = load_module('plot_multiseeds', '/content/project/plot_multiseeds.py')

from gcn import SimpleGNN, train_eval_loop_gnn_cora
from gat import GAT, train_eval_loop_gat
from linear_probe import linear_probing, linear_probing_auc
from my_datasets import load_dataset
from synthetic_dataset import load_synthetic_graph
from plot_multiseeds import _aggregate_probe_metric, _aggregate_trace_metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = [42, 123, 456, 789, 1024]
print(f"Device: {device}")


# 1. Load Datasets

In [ ]:
datasets = {}

for level, fname in [('high_homo_sy', 'highhomodata.pt'),
                     ('mid_homo_sy', 'midhomodata.pt'),
                     ('low_homo_sy', 'lowhomodata.pt')]:
    data = load_synthetic_graph(f'/content/project/graphuniverse/{fname}', device=device)
    data['edge_index'] = data['A'].nonzero(as_tuple=False).t().contiguous()
    datasets[level] = data

for name in ['cora', 'pubmed']:
    data = load_dataset(name, device=device)
    data['edge_index'] = data['A'].nonzero(as_tuple=False).t().contiguous()
    datasets[name] = data

for k, v in datasets.items():
    print(f"{k}: {v['X'].shape[0]} nodes, {v['X'].shape[1]} features, {v['num_classes']} classes")

# 2. Train GCN & GAT (1–10 layers, 5 seeds)

In [ ]:
# Cell 6: Train all models + probe
# ============================================================
configs = [
    ('high_homo_sy', 'gcn_h32_high_homo_sy', 'sgat4h_relu_high_homo_sy'),
    ('mid_homo_sy',  'gcn_h32_mid_homo_sy',  'sgat4h_relu_mid_homo_sy'),
    ('low_homo_sy',  'gcn_h32_low_homo_sy',  'sgat4h_relu_low_homo_sy'),
    ('cora',         'gcn_h32_cora',          'sgat4h_relu_cora'),
    ('pubmed',       'gcn_h32_pubmed',        'sgat4h_relu_pubmed'),
]

MAX_LAYERS = 10

for dataset_key, gcn_dir, gat_dir in configs:
    data = datasets[dataset_key]

    # ----- GCN -----
    print(f"\n{'='*50} GCN on {dataset_key} {'='*50}")
    probe_results = {seed: {} for seed in SEEDS}
    for seed in SEEDS:
        for nl in range(1, MAX_LAYERS + 1):
            torch.manual_seed(seed); np.random.seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.empty_cache()
            model = SimpleGNN(data['X'].shape[1], 32, data['num_classes'], nl, data['A']).to(device)
            train_eval_loop_gnn_cora(model, data['X'], data['train_y'], data['train_mask'],
                                     data['X'], data['valid_y'], data['valid_mask'],
                                     data['X'], data['test_y'], data['test_mask'])
            probe_results[seed][nl] = linear_probing(model, data['X'], data['labels'],
                                                      data['train_mask'], data['test_mask'], data['num_classes'])
            del model; torch.cuda.empty_cache()
            print(f"  GCN {nl}L seed={seed} done")

    os.makedirs(f'/content/project/results/{gcn_dir}', exist_ok=True)
    with open(f'/content/project/results/{gcn_dir}/probe_results.pkl', 'wb') as f:
        pickle.dump(probe_results, f)

    # ----- GAT -----
    print(f"\n{'='*50} GAT on {dataset_key} {'='*50}")
    probe_results = {seed: {} for seed in SEEDS}
    for seed in SEEDS:
        for nl in range(1, MAX_LAYERS + 1):
            torch.manual_seed(seed); np.random.seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.empty_cache()
            model = GAT(data['X'].shape[1], 8, data['num_classes'], nl,
                        data['edge_index'], num_heads=4).to(device)
            train_eval_loop_gat(model, data['X'], data['train_y'], data['train_mask'],
                                data['X'], data['valid_y'], data['valid_mask'],
                                data['X'], data['test_y'], data['test_mask'])
            probe_results[seed][nl] = linear_probing(model, data['X'], data['labels'],
                                                      data['train_mask'], data['test_mask'], data['num_classes'])
            del model; torch.cuda.empty_cache()
            print(f"  GAT {nl}L seed={seed} done")

    os.makedirs(f'/content/project/results/{gat_dir}', exist_ok=True)
    with open(f'/content/project/results/{gat_dir}/probe_results.pkl', 'wb') as f:
        pickle.dump(probe_results, f)

print("\nAll 1-10 layer training complete!")


# 3. Deep Models (16, 20 layers on Cora & PubMed)

In [ ]:
# Cell 8: Train deep models
# ============================================================
DEEP_DEPTHS = [16, 20]

for dataset_key, gcn_dir, gat_dir in [('cora', 'gcn_h32_cora', 'sgat4h_relu_cora'),
                                       ('pubmed', 'gcn_h32_pubmed', 'sgat4h_relu_pubmed')]:
    data = datasets[dataset_key]

    # Load existing and merge
    with open(f'/content/project/results/{gcn_dir}/probe_results.pkl', 'rb') as f:
        gcn_probe = pickle.load(f)
    with open(f'/content/project/results/{gat_dir}/probe_results.pkl', 'rb') as f:
        gat_probe = pickle.load(f)

    for nl in DEEP_DEPTHS:
        # GCN
        for seed in SEEDS:
            torch.manual_seed(seed); np.random.seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.empty_cache()
            model = SimpleGNN(data['X'].shape[1], 32, data['num_classes'], nl, data['A']).to(device)
            train_eval_loop_gnn_cora(model, data['X'], data['train_y'], data['train_mask'],
                                     data['X'], data['valid_y'], data['valid_mask'],
                                     data['X'], data['test_y'], data['test_mask'])
            gcn_probe[seed][nl] = linear_probing(model, data['X'], data['labels'],
                                                  data['train_mask'], data['test_mask'], data['num_classes'])
            del model; torch.cuda.empty_cache()
            print(f"  GCN {nl}L {dataset_key} seed={seed} done")

        # GAT
        for seed in SEEDS:
            torch.manual_seed(seed); np.random.seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.empty_cache()
            model = GAT(data['X'].shape[1], 8, data['num_classes'], nl,
                        data['edge_index'], num_heads=4).to(device)
            train_eval_loop_gat(model, data['X'], data['train_y'], data['train_mask'],
                                data['X'], data['valid_y'], data['valid_mask'],
                                data['X'], data['test_y'], data['test_mask'])
            gat_probe[seed][nl] = linear_probing(model, data['X'], data['labels'],
                                                  data['train_mask'], data['test_mask'], data['num_classes'])
            del model; torch.cuda.empty_cache()
            print(f"  GAT {nl}L {dataset_key} seed={seed} done")

    with open(f'/content/project/results/{gcn_dir}/probe_results.pkl', 'wb') as f:
        pickle.dump(gcn_probe, f)
    with open(f'/content/project/results/{gat_dir}/probe_results.pkl', 'wb') as f:
        pickle.dump(gat_probe, f)

print("Deep models done!")


# 4. AUC Experiments (entropy separability)

In [ ]:
# Cell 10: Run AUC probing
# ============================================================
for dataset_key, gcn_dir, gat_dir in configs:
    data = datasets[dataset_key]

    # GCN
    auc_results = {seed: {} for seed in SEEDS}
    for seed in SEEDS:
        for nl in range(1, MAX_LAYERS + 1):
            torch.manual_seed(seed); np.random.seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.empty_cache()
            model = SimpleGNN(data['X'].shape[1], 32, data['num_classes'], nl, data['A']).to(device)
            train_eval_loop_gnn_cora(model, data['X'], data['train_y'], data['train_mask'],
                                     data['X'], data['valid_y'], data['valid_mask'],
                                     data['X'], data['test_y'], data['test_mask'])
            auc_results[seed][nl] = linear_probing_auc(model, data['X'], data['labels'],
                                                        data['train_mask'], data['test_mask'], data['num_classes'])
            del model; torch.cuda.empty_cache()
    with open(f'/content/project/results/{gcn_dir}/auc_results.pkl', 'wb') as f:
        pickle.dump(auc_results, f)
    print(f"GCN AUC on {dataset_key} saved")

    # GAT
    auc_results = {seed: {} for seed in SEEDS}
    for seed in SEEDS:
        for nl in range(1, MAX_LAYERS + 1):
            torch.manual_seed(seed); np.random.seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.empty_cache()
            model = GAT(data['X'].shape[1], 8, data['num_classes'], nl,
                        data['edge_index'], num_heads=4).to(device)
            train_eval_loop_gat(model, data['X'], data['train_y'], data['train_mask'],
                                data['X'], data['valid_y'], data['valid_mask'],
                                data['X'], data['test_y'], data['test_mask'])
            auc_results[seed][nl] = linear_probing_auc(model, data['X'], data['labels'],
                                                        data['train_mask'], data['test_mask'], data['num_classes'])
            del model; torch.cuda.empty_cache()
    with open(f'/content/project/results/{gat_dir}/auc_results.pkl', 'wb') as f:
        pickle.dump(auc_results, f)
    print(f"GAT AUC on {dataset_key} saved")

print("All AUC experiments done!")


# 5. Figures (load from saved results)

In [ ]:
homo_configs = {
    'High (0.9)': ('gcn_h32_high_homo_sy', 'sgat4h_relu_high_homo_sy'),
    'Mid (0.5)':  ('gcn_h32_mid_homo_sy',  'sgat4h_relu_mid_homo_sy'),
    'Low (0.1)':  ('gcn_h32_low_homo_sy',  'sgat4h_relu_low_homo_sy'),
}
all_configs = {
    'Cora':       ('gcn_h32_cora',          'sgat4h_relu_cora'),
    'PubMed':     ('gcn_h32_pubmed',        'sgat4h_relu_pubmed'),
    **homo_configs,
}
colors_homo = {'High (0.9)': 'darkgreen', 'Mid (0.5)': 'orange', 'Low (0.1)': 'red'}

def load_probe(dir_name):
    with open(f'/content/project/results/{dir_name}/probe_results.pkl', 'rb') as f:
        return pickle.load(f)

def load_auc(dir_name):
    with open(f'/content/project/results/{dir_name}/auc_results.pkl', 'rb') as f:
        return pickle.load(f)

def compute_auc_per_layer(auc_data, seeds, num_layers):
    all_aucs = []
    for seed in seeds:
        layer_aucs = []
        for layer_data in auc_data[seed][num_layers]:
            h = layer_data['entropy_per_node']
            mask = layer_data['per_layer_correct_mask']
            hc, hi = h[mask], h[~mask]
            if len(hc) == 0 or len(hi) == 0:
                layer_aucs.append(0.5); continue
            auc = sum((x > hc).sum() + 0.5 * (x == hc).sum() for x in hi) / (len(hi) * len(hc))
            layer_aucs.append(auc)
        all_aucs.append(layer_aucs)
    all_aucs = np.array(all_aucs)
    return np.arange(1, all_aucs.shape[1]), all_aucs[:, 1:].mean(0), all_aucs[:, 1:].std(0)

def compute_spearman_per_layer(auc_data, seeds, num_layers):
    all_rhos = []
    for seed in seeds:
        layer_rhos = []
        for layer_data in auc_data[seed][num_layers]:
            rho, _ = spearmanr(layer_data['entropy_per_node'], layer_data['neg_log_p_true_per_node'])
            layer_rhos.append(rho)
        all_rhos.append(layer_rhos)
    all_rhos = np.array(all_rhos)
    return np.arange(1, all_rhos.shape[1]), all_rhos[:, 1:].mean(0), all_rhos[:, 1:].std(0)

def compute_per_node_spearman(auc_data, seeds, num_layers):
    all_mean_rhos = []
    for seed in seeds:
        layers_data = auc_data[seed][num_layers]
        n_nodes = len(layers_data[0]['entropy_per_node'])
        n_layers = len(layers_data) - 1
        if n_layers < 3: all_mean_rhos.append(np.nan); continue
        H = np.zeros((n_nodes, n_layers))
        NLL = np.zeros((n_nodes, n_layers))
        for k in range(n_layers):
            H[:, k] = layers_data[k+1]['entropy_per_node']
            NLL[:, k] = layers_data[k+1]['neg_log_p_true_per_node']
        node_rhos = []
        for v in range(n_nodes):
            if np.std(H[v]) < 1e-10 or np.std(NLL[v]) < 1e-10: continue
            rho, _ = spearmanr(H[v], NLL[v])
            if not np.isnan(rho): node_rhos.append(rho)
        all_mean_rhos.append(np.mean(node_rhos) if node_rhos else np.nan)
    return np.nanmean(all_mean_rhos), np.nanstd(all_mean_rhos)
